# Binary Cross-Entropy Loss in Binary Classification

In [1]:
import torch
import torch.nn as nn

# Initialize prediction values (already in probability form after sigmoid) and actual labels
y_pred_prob = torch.tensor([0.8, 0.1, 0.6], requires_grad=True)  # Predicted probabilities
y_true = torch.tensor([1.0, 0.0, 1.0])  # Actual labels

# Initialize BCE loss function
criterion = nn.BCELoss()
loss = criterion(y_pred_prob, y_true)

print(f'Binary Cross-Entropy Loss: {loss.item()}')

# Backward pass
loss.backward()

Binary Cross-Entropy Loss: 0.27977654337882996


In [2]:
import torch
import torch.nn as nn

# Initialize prediction values in logits form and actual labels
y_pred_logits = torch.tensor([1.5, -1.0, 0.2], requires_grad=True)  # Predicted logits
y_true = torch.tensor([1.0, 0.0, 1.0])  # Actual labels

# Initialize BCE with Logits loss function
criterion = nn.BCEWithLogitsLoss()
loss = criterion(y_pred_logits, y_true)

print(f'Binary Cross-Entropy with Logits Loss: {loss.item()}')

# Backward pass
loss.backward()

Binary Cross-Entropy with Logits Loss: 0.3709379732608795


# Hinge Loss

In [3]:
import torch
import torch.optim as optim

# Sample Data
y_pred = torch.tensor([0.9, -0.1, 0.8, -0.4], requires_grad=True)  # Predictions
y_true = torch.tensor([1, -1, 1, -1], dtype=torch.float32)         # Ground Truths (-1 or 1)

# Define Hinge Loss
def hinge_loss(y_pred, y_true):
    return torch.mean(torch.clamp(1 - y_pred * y_true, min=0))

# Compute loss
loss = hinge_loss(y_pred, y_true)
print("Hinge Loss:", loss.item())

# Optimize using Gradient Descent
optimizer = optim.SGD([y_pred], lr=0.1)
optimizer.zero_grad()
loss.backward()
optimizer.step()

Hinge Loss: 0.45000001788139343


# Cross-Entropy Loss

In [5]:
import torch

# Assume probability distributions
# f(x): Predicted distribution from the model
f_x = torch.tensor([0.4, 0.1, 0.3, 0.2])

# p(x): True distribution (one-hot vector)
p_x = torch.tensor([0.0, 1.0, 0.0, 0.0])

# Calculate Cross-Entropy Loss manually
cross_entropy_loss = -torch.sum(p_x * torch.log(f_x + 1e-10))
print("Manual Cross-Entropy Loss:", cross_entropy_loss.item())

Manual Cross-Entropy Loss: 2.3025851249694824


# Sparse Categorical Cross Entropy (nn.CrossEntropyLoss in PyTorch)

Chú ý: Trong nn.CrossEntropyLoss( ) đã chứa hàm softmax, nên giá trị của output không nhất thiết phải nằm trong khoảng [0;1]

In [8]:
import torch
import torch.nn as nn

# Example logits for a 3-class classification (batch size = 2)
logits = torch.tensor([[2.0, 1.0, 0.1], [0.5, 2.5, 0.3]])

# True class labels
labels = torch.tensor([0, 2])

# Initialize Cross-Entropy Loss
criterion = nn.CrossEntropyLoss()

# Calculate the Cross-Entropy Loss
loss = criterion(logits, labels)

print(f"Cross-Entropy Loss: {loss.item()}")

Cross-Entropy Loss: 1.4185397624969482


# Label Smoothing

In [9]:
import torch
import torch.nn.functional as F

class LabelSmoothingLoss(torch.nn.Module):
    def __init__(self, classes: int, smoothing: float = 0.1):
        super(LabelSmoothingLoss, self).__init__()
        self.classes = classes
        self.smoothing = smoothing

    def forward(self, pred, target):
        # Convert target labels to one-hot encoding
        target_one_hot = F.one_hot(target, num_classes=self.classes).float()

        # Apply Label Smoothing
        target_smooth = target_one_hot * (1 - self.smoothing) + self.smoothing / self.classes

        # Calculate cross-entropy loss with the smoothed probabilities
        log_prob = F.log_softmax(pred, dim=-1)
        loss = -torch.sum(log_prob * target_smooth, dim=-1)

        return loss.mean()

# Example usage
# Assume we have a batch of predictions and true labels
num_classes = 5
label_smoothing = 0.1
batch_size = 3

# Initialize the loss function with Label Smoothing
criterion = LabelSmoothingLoss(classes=num_classes, smoothing=label_smoothing)

# Assume predictions (logits) and labels
pred = torch.randn(batch_size, num_classes)  # Unnormalized values (logits)
target = torch.tensor([1, 0, 3])             # Ground truth labels

# Calculate loss
loss = criterion(pred, target)
print(f'Label Smoothing Loss: {loss.item()}')

Label Smoothing Loss: 1.875360131263733


# Binary Cross-Entropy Loss in Multi-label Classification

In [13]:
import torch
import torch.nn as nn

# Number of samples and labels
num_samples = 4
num_labels = 3

# Model predictions in logits form (not passed through sigmoid)
y_pred_logits = torch.tensor([[0.8, -1.2, 1.5],
                              [1.2, 0.3, -0.8],
                              [0.2, 2.0, -1.5],
                              [0.7, -0.5, 1.0]])

# Ground truth labels, each row represents labels for a sample
# (1 indicates the label is present, 0 indicates the label is absent)
y_true = torch.tensor([[1, 0, 1],
                       [0, 1, 0],
                       [1, 1, 0],
                       [0, 0, 1]]).float()

# Initialize BCE with Logits Loss
criterion = nn.BCEWithLogitsLoss()

# Calculate BCE Loss
loss = criterion(y_pred_logits, y_true)
print(f"Binary Cross-Entropy Loss: {loss.item()}")

Binary Cross-Entropy Loss: 0.5034615993499756


# Pairwise Ranking Loss

In [10]:
import torch
import torch.nn as nn

torch.manual_seed(0)
num_samples = 10
num_labels = 5
X = torch.randn(num_samples, 4)
Y = (torch.rand(num_samples, num_labels) > 0.5).float()

class SimpleLinearModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(SimpleLinearModel, self).__init__()
        self.fc = nn.Linear(input_dim, output_dim)

    def forward(self, x):
        return self.fc(x)

model = SimpleLinearModel(input_dim=4, output_dim=num_labels)

In [12]:
def pairwise_ranking_loss(y_pred, y_true):
    loss = 0
    num_pairs = 0

    for i in range(y_pred.size(0)):  # Iterate over each sample
        pos_indices = torch.where(y_true[i] == 1)[0]  # Positive labels
        neg_indices = torch.where(y_true[i] == 0)[0]  # Negative labels

        for pos in pos_indices:
            for neg in neg_indices:
                loss += torch.clamp(1 - y_pred[i, pos] + y_pred[i, neg], min=0)
                num_pairs += 1

    if num_pairs > 0:
        loss /= num_pairs

    return loss

y_pred = model(X)
loss = pairwise_ranking_loss(y_pred, Y)
print(f'Pairwise Ranking Loss: {loss.item()}')

Pairwise Ranking Loss: 0.741832435131073


# BCE Loss + PR Loss

In [14]:
import torch
import torch.nn as nn

# Re-define pairwise_ranking_loss if needed (assuming it is defined in previous code)
def pairwise_ranking_loss(y_pred, y_true):
    loss = 0
    num_pairs = 0

    for i in range(y_pred.size(0)):  # Iterate over each sample
        pos_indices = torch.where(y_true[i] == 1)[0]  # Positive labels
        neg_indices = torch.where(y_true[i] == 0)[0]  # Negative labels

        for pos in pos_indices:
            for neg in neg_indices:
                loss += torch.clamp(1 - y_pred[i, pos] + y_pred[i, neg], min=0)
                num_pairs += 1

    if num_pairs > 0:
        loss /= num_pairs

    return loss

# Define BCE loss function
bce_loss_fn = nn.BCEWithLogitsLoss()

# Combined loss function
def combined_loss(y_pred, y_true, alpha=0.5):
    bce_loss = bce_loss_fn(y_pred, y_true)
    pr_loss = pairwise_ranking_loss(y_pred, y_true)
    total_loss = alpha * bce_loss + (1 - alpha) * pr_loss
    return total_loss

# Sample data for testing
torch.manual_seed(0)
num_samples = 4
num_labels = 3

# Predicted logits and true labels
y_pred = torch.randn(num_samples, num_labels)  # Random logits
y_true = torch.randint(0, 2, (num_samples, num_labels)).float()  # Random binary labels

# Print test data
print("Predicted logits (y_pred):\n", y_pred)
print("True labels (y_true):\n", y_true)

# Calculate individual losses
bce_loss = bce_loss_fn(y_pred, y_true)
pr_loss = pairwise_ranking_loss(y_pred, y_true)
combined = combined_loss(y_pred, y_true, alpha=0.5)

# Print results
print(f"BCE Loss: {bce_loss.item()}")
print(f"Pairwise Ranking Loss: {pr_loss.item()}")
print(f"Combined Loss (alpha=0.5): {combined.item()}")

Predicted logits (y_pred):
 tensor([[ 1.5410, -0.2934, -2.1788],
        [ 0.5684, -1.0845, -1.3986],
        [ 0.4033,  0.8380, -0.7193],
        [-0.4033, -0.5966,  0.1820]])
True labels (y_true):
 tensor([[0., 1., 1.],
        [1., 1., 0.],
        [1., 0., 1.],
        [0., 1., 1.]])
BCE Loss: 0.9912829995155334
Pairwise Ranking Loss: 1.730001449584961
Combined Loss (alpha=0.5): 1.3606421947479248
